# Single-GPU Embedding Engine


In [1]:
!pip install -q datasets transformers sentence-transformers faiss-gpu pandas numpy tqdm scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 MB 13.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 100.3 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is inc

In [38]:
#libraries
import os
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import faiss
from datasets import load_dataset

In [4]:
#GPU check
print("pytorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range( torch.cuda.device_count()):
        print(f"GPU{i}:",torch.cuda.get_device_name(i))

else:
    print("No GPU detected")

pytorch version: 2.10.0+cu128
CUDA available: True
GPU count: 2
GPU0: Tesla T4
GPU1: Tesla T4


# Set device and config

In [5]:
device ="cuda" if torch.cuda.is_available() else "cpu"
gpu_count= torch.cuda.device_count()

if gpu_count>=1:
    data_size=10000
    batch_size=64
else:
    data_size=1000
    batch_size=16

print("device:", device)
print("gpu count:", gpu_count)
print("batch_size:",batch_size)
print("Data_size:",data_size)

device: cuda
gpu count: 2
batch_size: 64
Data_size: 10000


In [6]:
#Load MS marco dataset

corpus= load_dataset(
    "sentence-transformers/msmarco",
    "corpus",
    split=f"train[:{data_size}]"
)

corpus

README.md: 0.00B [00:00, ?B/s]

corpus/train-00000-of-00007.parquet:   0%|          | 0.00/238M [00:00<?, ?B/s]

corpus/train-00001-of-00007.parquet:   0%|          | 0.00/240M [00:00<?, ?B/s]

corpus/train-00002-of-00007.parquet:   0%|          | 0.00/242M [00:00<?, ?B/s]

corpus/train-00003-of-00007.parquet:   0%|          | 0.00/243M [00:00<?, ?B/s]

corpus/train-00004-of-00007.parquet:   0%|          | 0.00/245M [00:00<?, ?B/s]

corpus/train-00005-of-00007.parquet:   0%|          | 0.00/241M [00:00<?, ?B/s]

corpus/train-00006-of-00007.parquet:   0%|          | 0.00/240M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8841823 [00:00<?, ? examples/s]

Dataset({
    features: ['passage_id', 'passage'],
    num_rows: 10000
})

In [7]:
print(corpus[0])

{'passage_id': '0', 'passage': 'The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.'}


In [8]:
# convert to DataFrame and clean
df= corpus.to_pandas()

print("origional shape:",df.shape)
print(df.columns)
df=df.dropna(subset=["passage"])
df=df.drop_duplicates(subset=["passage"])
df=df.reset_index(drop=True)

print("Cleaned shape:",df.shape)
df.head()

origional shape: (10000, 2)
Index(['passage_id', 'passage'], dtype='object')
Cleaned shape: (10000, 2)


,passage_id,passage
0,0,The presence of communication amid scientific ...
1,1,The Manhattan Project and its atomic bomb help...
2,2,Essay on The Manhattan Project - The Manhattan...
3,3,The Manhattan Project was the name for a proje...
4,4,versions of each volume as well as complementa...


In [9]:
#load transformer embedding model
model_name="sentence-transformers/all-MiniLM-L6-v2"

tokenizer= AutoTokenizer.from_pretrained(model_name)
model=AutoModel.from_pretrained(model_name)

model.to(device)
model.eval()

print("model loaded:",model_name)
print("running on", device)


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model loaded: sentence-transformers/all-MiniLM-L6-v2
running on cuda


In [34]:
# mean pooling function
def mean_pooling(model_output,attention_mask):
    token_embeddings=model_output.last_hidden_state
    input_mask_expanded= attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sentence_embedding= torch.sum(token_embeddings*input_mask_expanded,dim=1)/torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    return sentence_embedding
    

In [35]:
# pytorch dataset
class passagedataset(Dataset):
    def __init__(self, passages,passage_ids, tokenizer, max_length=256):
        self.passages= passages
        self.passage_ids= passage_ids
        self.tokenizer= tokenizer
        self.max_length=max_length

    def __len__(self):
        return len(self.passages)

    def __getitem__(self,idx):
        text=str(self.passages[idx])
        
        encoded = self.tokenizer(text,
                                padding="max_length",
                                truncation=True,
                                 max_length=self.max_length,
                                 return_tensors="pt"
                                )
        return {
            "passage_id": self.passage_ids[idx],
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0)
        }
    

In [36]:
#Create DataLoader
passage_dataset=passagedataset(
    passages=df["passage"].tolist(),
    passage_ids=df["passage_id"].tolist(),
    tokenizer=tokenizer,
    max_length=256
)
passage_loader=DataLoader(
    passage_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)
# debugging
print("Total passages:", len(passage_dataset))
print("total batches:",len(passage_loader))

Total passages: 10000
total batches: 157


In [39]:
# Generate embeddings
all_embeddings=[]
all_passage_ids=[]

start_time=time.time()

with torch.no_grad():
    for batch in tqdm(passage_loader, desc="Generating embeddings"):
        input_ids=batch["input_ids"].to(device)
        attention_mask=batch["attention_mask"].to(device)
        
        outputs= model(input_ids=input_ids,attention_mask=attention_mask)
        embeddings=mean_pooling(outputs,attention_mask)
        
        embeddings=F.normalize(embeddings,p=2,dim=1)
        all_embeddings.append(embeddings.cpu().numpy())
        all_passage_ids.extend(batch["passage_id"])

end_time=time.time()

embeddings_matrix= np.vstack(all_embeddings).astype("float32")

print("Embedding matrix shape:", embeddings_matrix.shape)
print("Total passage IDS:", len(all_passage_ids))
print("time taken:",round(end_time- start_time,2),"seconds")
print("Throughput:",round(len(df)/(end_time-start_time),2),"passages/sec")


Generating embeddings:   0%|          | 0/157 [00:00<?, ?it/s]

Embedding matrix shape: (10000, 384)
Total passage IDS: 10000
time taken: 28.27 seconds
Throughput: 353.79 passages/sec


In [41]:
# BUILD FAISS index

embedding_dim= embeddings_matrix.shape[1]

index=faiss.IndexFlatIP(embedding_dim)
index.add(embeddings_matrix)

print("embedding dimension:", embedding_dim)
print("Total vectors in FAISS:", index.ntotal)





embedding dimension: 384
Total vectors in FAISS: 10000


In [46]:
# query embedding function
def embed_query(query):
    encoded=tokenizer(
        query,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors='pt'
    )
    input_ids=encoded["input_ids"].to(device)
    attention_mask=encoded["attention_mask"].to(device)
    with torch.no_grad():
        outputs=model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        query_embedding=mean_pooling(outputs,attention_mask)
        query_embedding= F.normalize(query_embedding,p=2,dim=1)

    return query_embedding.cpu().numpy().astype("float32")
        

In [47]:
q_vec = embed_query("what are the symptoms of diabetes")
print(q_vec.shape)
print(q_vec.dtype)

(1, 384)
float32


In [50]:
# semantic search function
def semantic_search(query,top_k=5):
    query_vector=embed_query(query)
    scores,indices= index.search(query_vector,top_k)
    results=[]
    for score,idx in zip(scores[0],indices[0]):
        row=df.iloc[idx]
        results.append({
            "passage_id": row["passage_id"],
            "score": float(score),
            "passage": row["passage"]
        })

    return results

In [51]:
# test search
query = "what are the symptoms of diabetes"

results = semantic_search(query, top_k=5)

for rank, result in enumerate(results, start=1):
    print(f"Rank {rank}")
    print("Passage ID:", result["passage_id"])
    print("Score:", round(result["score"], 4))
    print("Passage:", result["passage"][:700])
    print("-" * 100)

Rank 1
Passage ID: 6052
Score: 0.5163
Passage: If you are asking what blood glucose level is considered dangerous; then, you should know that it all depends on your current health situation. However, the dangerous sugar levels, where the symptoms of hypoglycemia will occur, are considered those less than 40 mg/dL. Shaking, nausea, tremors, sweating or increased heart palpitations are some symptoms to recognize your blood sugar is dropping.
----------------------------------------------------------------------------------------------------
Rank 2
Passage ID: 6058
Score: 0.505
Passage: Random check. The doctor tests your blood sugar and itâs higher than 200, plus youâre peeing more, always thirsty, and youâve gained or lost a significant amount of weight. Heâll then do a fasting sugar level test or an oral glucose tolerance test to confirm the diagnosis.
----------------------------------------------------------------------------------------------------
Rank 3
Passage ID: 6055
Sc

In [52]:
#benchmark

benchmark_queries=[
    "what are the symptoms of diabetes",
    "how to improve computer performance",
    "what causes high blood pressure",
    "how does solar energy work",
    "best way to learn machine learning",
    "what is credit car fraud",
    "how to lose weight safely",
    "what is cloud computing",
    "how to treat back pain",
    "benefits of regular exercise"
]
latencies=[]
for query in benchmark_queries:
    start=time.time()
    _=semantic_search(query,top_k=5)
    end=time.time()
    latencies.append(end-start)

print("Total queries:", len(benchmark_queries))
print("Average search latency:", round(np.mean(latencies) * 1000, 2), "ms")
print("Minimum latency:", round(np.min(latencies) * 1000, 2), "ms")
print("Maximum latency:", round(np.max(latencies) * 1000, 2), "ms")

Total queries: 10
Average search latency: 7.12 ms
Minimum latency: 5.83 ms
Maximum latency: 10.64 ms


In [53]:
#check whether the model uses single GPU or multiple GPU (we have 2 GPU available)
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_name(1))

0
Tesla T4
Tesla T4
